# C3 — Task-Conditioned Metadata Selection

Evaluate whether assigning each question to its *optimal* Les Débats metadata strategy
(learned from a training split) improves retrieval over the default baseline, mirroring
the chunking approach in **C2** but operating on the metadata filter/reranking axis
studied in **A6_NER_strat_comparison**.

**Metadata strategy types (14 total):**
- **Filter** (7): pre-filters Les Débats by `doc_type`, `speaker_count`, or `doc_length` at query time
- **Rerank** (3): semantic retrieval then re-scores based on metadata fields
- **Hybrid** (4): filter query + rerank combined

**Pipeline:**
1. Load questions + stored embeddings from ChromaDB (local — **zero API calls**)
2. 75/25 train/test split (seed=42)
3. Cluster training questions: UMAP(10D) → HDBSCAN
4. For each cluster × metadata strategy × routing, compute Coverage@3 on training questions
   *(results cached in `data/METADATA_results_NER/` — re-querying local ChromaDB, run once)*
5. Per-routing: best metadata strategy per cluster
6. Assign test questions to nearest training cluster → optimal metadata strategy
7. Evaluate 8 conditions (4 routing × 2 metadata policies: Global Baseline / Task-Conditioned)

**Key principle:** each TC condition uses the strategy map *trained under the same routing*
applied at test time, eliminating the mismatch identified and fixed in C2.

## 1. Imports & Configuration

In [1]:
import json
import os
import sys

import hdbscan
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import umap
from sklearn.metrics.pairwise import euclidean_distances
from tqdm import tqdm

if os.path.basename(os.getcwd()) == 'notebooks':
    sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from scripts.config import COHERE_API_KEY, EMBEDDINGS_1887_COHERE
from scripts.utils.chroma_utils import load_chroma_collection
from scripts.retrieval_utils import (
    apply_no_rerouting,
    apply_query_rerouting,
    apply_forced_multicollection,
    apply_forced_equal_from_all_collections,
    compute_recall_metrics_dataframe,
    rerank_type_boost,
    rerank_speaker_boost,
    rerank_combined,
)
from scripts.evaluation_utils import compute_coverage_row_list

print('Imports OK')

c:\Users\plathoon\AppData\Local\anaconda3\envs\debattre\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


In [2]:
SEED = 42
TRAIN_RATIO = 0.75
K_VALUES = [3, 5, 10]
SCORE_K = 3  # metric used for cluster-strategy selection

BASE_DIR = os.getcwd() if os.path.basename(os.getcwd()) != 'notebooks' else os.path.join(os.getcwd(), '..')
DATA_DIR = os.path.join(BASE_DIR, 'data')
CHUNKING_CACHE_DIR = os.path.join(DATA_DIR, 'RETRIEVER_results_chunking_comparison')
META_CACHE_DIR = os.path.join(DATA_DIR, 'METADATA_results_NER')
os.makedirs(META_CACHE_DIR, exist_ok=True)

BASELINE_METADATA = 'baseline'
BASELINE_CHUNKING = '10000_hierarchical'
N_PER_COLLECTION = 15

CLASS_TO_COLLECTIONS = {
    "Le Gaulois + L'Intransigeant": ['Le Gaulois', "L'Intransigeant"],
    "L'Intransigeant + Les D\u00e9bats": ['Le Gaulois', "L'Intransigeant", 'Les D\u00e9bats'],
    "Le Gaulois + Les D\u00e9bats":      ['Le Gaulois', "L'Intransigeant", 'Les D\u00e9bats'],
}

ROUTING_CONFIGS = {
    'no_rerouting': {
        'fn': apply_no_rerouting,
        'kwargs': {},
        'docs_col': 'documents',
        'sources_col': 'sources',
        'label': 'No Rerouting',
    },
    'qre': {
        'fn': apply_query_rerouting,
        'kwargs': {'class_to_collections': CLASS_TO_COLLECTIONS},
        'docs_col': 'documents_rerouted',
        'sources_col': 'sources_rerouted',
        'label': 'Query Rerouting (QRe)',
    },
    'ums': {
        'fn': apply_forced_equal_from_all_collections,
        'kwargs': {},
        'docs_col': 'documents_forced_equal',
        'sources_col': 'sources_forced_equal',
        'label': 'UMS (Forced Equal)',
    },
    'qre_ums': {
        'fn': apply_forced_multicollection,
        'kwargs': {'class_to_collections': CLASS_TO_COLLECTIONS},
        'docs_col': 'documents_forced',
        'sources_col': 'sources_forced',
        'label': 'QRe + UMS',
    },
}

NOISY_DOC_TYPES = [
    'vote_list', 'absence_list', 'fragment', 'chamber_header',
    'scrutin', 'excuse', 'conge', 'compte_rendu_header', 'tirage_sort',
]
DEBATE_TYPES = ['debate', 'legal_text', 'law_presentation', 'law_adoption', 'petition', 'sommaire']

# Each entry: (where_filter_or_None, needs_metadatas_bool, rerank_fn_or_None)
# needs_metadatas=True triggers include=['metadatas'] in the ChromaDB query
FILTER_STRATEGIES = {
    'baseline':                  (None, False, None),
    'exclude_noisy':             ({'doc_type': {'$nin': NOISY_DOC_TYPES}}, False, None),
    'debates_only':              ({'doc_type': 'debate'}, False, None),
    'debates_and_legal':         ({'doc_type': {'$in': DEBATE_TYPES}}, False, None),
    'min_speakers_2':            ({'speaker_count': {'$gte': 2}}, False, None),
    'min_length_500':            ({'doc_length': {'$gte': 500}}, False, None),
    'exclude_noisy_min_length':  (
        {'$and': [{'doc_type': {'$nin': NOISY_DOC_TYPES}}, {'doc_length': {'$gte': 200}}]},
        False, None,
    ),
    'rerank_type_boost':         (None, True, rerank_type_boost),
    'rerank_speaker_boost':      (None, True, rerank_speaker_boost),
    'rerank_combined':           (None, True, rerank_combined),
    'exclude_noisy__rerank_speaker': (
        {'doc_type': {'$nin': NOISY_DOC_TYPES}}, True, rerank_speaker_boost,
    ),
    'debates_legal__rerank_speaker': (
        {'doc_type': {'$in': DEBATE_TYPES}}, True, rerank_speaker_boost,
    ),
    'debates_legal__rerank_combined': (
        {'doc_type': {'$in': DEBATE_TYPES}}, True, rerank_combined,
    ),
    'triple_filter__rerank_combined': (
        {'$and': [
            {'doc_type': {'$in': DEBATE_TYPES}},
            {'speaker_count': {'$gte': 2}},
            {'doc_length': {'$gte': 200}},
        ]},
        True, rerank_combined,
    ),
}

METADATA_STRATEGY_IDS = list(FILTER_STRATEGIES.keys())  # 14 strategies

print(f'Meta cache dir: {META_CACHE_DIR}')
print(f'Metadata strategies ({len(METADATA_STRATEGY_IDS)}): {METADATA_STRATEGY_IDS}')
print(f'Routing configs: {list(ROUTING_CONFIGS.keys())}')

Meta cache dir: c:\Users\plathoon\Desktop\Aurelien\debattre-generate-questions\data\METADATA_results_NER
Metadata strategies (14): ['baseline', 'exclude_noisy', 'debates_only', 'debates_and_legal', 'min_speakers_2', 'min_length_500', 'exclude_noisy_min_length', 'rerank_type_boost', 'rerank_speaker_boost', 'rerank_combined', 'exclude_noisy__rerank_speaker', 'debates_legal__rerank_speaker', 'debates_legal__rerank_combined', 'triple_filter__rerank_combined']
Routing configs: ['no_rerouting', 'qre', 'ums', 'qre_ums']


## 2. Load Questions + Embeddings from ChromaDB

In [4]:
collection_questions = load_chroma_collection(
    EMBEDDINGS_1887_COHERE, api_key=COHERE_API_KEY, collection_name='questions_MH_v1'
)
result = collection_questions.get(include=['metadatas', 'documents', 'embeddings'])

questions_df = pd.DataFrame({
    'question':   result['documents'],
    'embeddings': list(result['embeddings']),
    'metadata':   result['metadatas'],
})
questions_df = pd.concat(
    [questions_df.drop('metadata', axis=1), questions_df['metadata'].apply(pd.Series)],
    axis=1,
)
questions_df = questions_df.loc[:, ~questions_df.columns.duplicated()]
questions_df = questions_df[questions_df['question'] != 'AUCUNE QUESTION'].reset_index(drop=True)

if 'gold_ids' in questions_df.columns and isinstance(questions_df['gold_ids'].iloc[0], str):
    questions_df['gold_ids'] = questions_df['gold_ids'].apply(json.loads)

# Embedding lookup dict — used when re-querying ChromaDB with metadata filters
embedding_by_uid = dict(zip(questions_df['unique_id'], questions_df['embeddings'].tolist()))

print(f'Loaded {len(questions_df)} questions')
print(f'Columns: {questions_df.columns.tolist()}')

Loaded 875 questions
Columns: ['question', 'embeddings', 'gold_ids', 'question_type', 'id_2', 'question_index', 'source_1', 'unique_id', 'id_1', 'source_2']


## 3. Train / Test Split (75/25, seed=42)

In [5]:
rng = np.random.default_rng(SEED)
shuffled_idx = rng.permutation(len(questions_df))
n_train = int(len(questions_df) * TRAIN_RATIO)

train_ids = set(questions_df.iloc[shuffled_idx[:n_train]]['unique_id'].tolist())
test_ids  = set(questions_df.iloc[shuffled_idx[n_train:]]['unique_id'].tolist())

train_df = questions_df[questions_df['unique_id'].isin(train_ids)].reset_index(drop=True)
test_df  = questions_df[questions_df['unique_id'].isin(test_ids)].reset_index(drop=True)

print(f'Train: {len(train_df)}  |  Test: {len(test_df)}')

Train: 656  |  Test: 219


## 4. Cluster Training Questions (UMAP → HDBSCAN)

Identical setup to C2 — 10-D UMAP reduction + HDBSCAN with the same seed,
so clusters are directly comparable across the two notebooks.

In [6]:
X_train = np.array(train_df['embeddings'].tolist())

CLUSTER_DIM = 10
reducer = umap.UMAP(n_components=CLUSTER_DIM, metric='cosine', random_state=SEED)
X_train_umap = reducer.fit_transform(X_train)

clusterer = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=5, metric='euclidean')
train_labels = clusterer.fit_predict(X_train_umap)

n_clusters = len(set(train_labels) - {-1})
n_noise    = (train_labels == -1).sum()
print(f'HDBSCAN \u2192 {n_clusters} clusters, {n_noise} noise points')

train_df = train_df.copy()
train_df['cluster'] = train_labels

# Centroids — include the HDBSCAN noise cluster (-1) so it competes equally
# (parity with C2: every training question is assigned to a real cluster centroid).
centroids = {}
for cid in set(train_labels):
    mask = train_labels == cid
    centroids[cid] = X_train_umap[mask].mean(axis=0)

cluster_train_qids = {
    cid: set(train_df[train_df['cluster'] == cid]['unique_id'].tolist())
    for cid in centroids
}

print(f'Centroids computed for {len(centroids)} clusters (including noise cluster -1)')


c:\Users\plathoon\AppData\Local\anaconda3\envs\debattre\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


HDBSCAN → 20 clusters, 123 noise points
Centroids computed for 21 clusters (including noise cluster -1)


## 5. Build / Cache Metadata Strategy DataFrames

For each strategy we re-query the **Les Débats** ChromaDB collection
(`debattre_1887_10000_hierarchical_cohere`) with an optional `where` filter
and/or apply a rerank function.  Newspaper results (Le Gaulois,
L'Intransigeant) are **unchanged** from the universal baseline JSONL.

All embeddings are already stored locally — **no new API calls**.  Results
are cached to `data/METADATA_results_NER/` so this section only runs once.

**Column schema** (same as chunking JSONL so routing functions work unchanged):  
`unique_id | gold_ids | predicted_class | retrieved_ids | distances | documents | sources`

In [7]:
# Load gold-lookup collections (same 3 as C2)
GOLD_COLLECTION_CONFIGS = [
    {'name': 'legaulois_1887_v1',                       'label': 'Le Gaulois'},
    {'name': 'lintransigeant_1887_v1',                  'label': "L'Intransigeant"},
    {'name': 'debattre_1887_10000_hierarchical_cohere', 'label': 'Les D\u00e9bats'},
]

gold_collections = {}
for cfg in GOLD_COLLECTION_CONFIGS:
    col = load_chroma_collection(
        EMBEDDINGS_1887_COHERE, api_key=COHERE_API_KEY, collection_name=cfg['name']
    )
    gold_collections[cfg['label']] = col
    print(f"  Loaded {cfg['label']} ({col.count()} docs)")

debat_collection = gold_collections['Les D\u00e9bats']
print('Gold collections ready.')

  Loaded Le Gaulois (78 docs)
  Loaded L'Intransigeant (79 docs)
  Loaded Les Débats (5103 docs)
Gold collections ready.


In [8]:
def load_universal_baseline_df():
    """Load the pre-cached universal JSONL for the baseline chunking strategy."""
    path = os.path.join(CHUNKING_CACHE_DIR, f'universal_{BASELINE_CHUNKING}.jsonl')
    with open(path, encoding='utf-8') as f:
        rows = [json.loads(line) for line in f if line.strip()]
    df = pd.DataFrame(rows)
    for col in ['gold_ids', 'retrieved_ids', 'distances', 'documents', 'sources']:
        if col in df.columns and len(df) > 0 and isinstance(df[col].iloc[0], str):
            df[col] = df[col].apply(json.loads)
    return df


def build_metadata_df(strat_name, where_filter, needs_metadatas, rerank_fn, base_df):
    """
    Re-query Les Débats with optional WHERE filter and/or reranking.
    Newspaper results are unchanged from base_df.
    """
    include_fields = ['documents', 'distances']
    if needs_metadatas or rerank_fn is not None:
        include_fields.append('metadatas')

    rows = []
    for _, row in tqdm(base_df.iterrows(), total=len(base_df), desc=strat_name):
        # ---- Newspaper part: keep as-is from base_df ----
        np_ids, np_dists, np_docs, np_srcs, np_metas = [], [], [], [], []
        for i, src in enumerate(row['sources']):
            if src != 'Les D\u00e9bats':
                np_ids.append(row['retrieved_ids'][i])
                np_dists.append(row['distances'][i])
                np_docs.append(row['documents'][i])
                np_srcs.append(src)
                np_metas.append(None)  # no metadata needed for newspapers

        # ---- Les Débats part: re-query with optional filter ----
        uid = row['unique_id']
        emb = embedding_by_uid.get(uid)

        if emb is None:
            # Fallback: keep existing Les Débats rows from base_df
            db_ids   = [rid for rid, src in zip(row['retrieved_ids'], row['sources']) if src == 'Les D\u00e9bats']
            db_dists = [d   for d,   src in zip(row['distances'],     row['sources']) if src == 'Les D\u00e9bats']
            db_docs  = [doc for doc, src in zip(row['documents'],     row['sources']) if src == 'Les D\u00e9bats']
            db_metas = [None] * len(db_ids)
            db_srcs  = ['Les D\u00e9bats'] * len(db_ids)
        else:
            query_kwargs = {
                'query_embeddings': [list(emb)],
                'n_results': N_PER_COLLECTION,
                'include': include_fields,
            }
            if where_filter is not None:
                query_kwargs['where'] = where_filter
            try:
                res = debat_collection.query(**query_kwargs)
                db_ids   = res['ids'][0]
                db_dists = res['distances'][0]
                db_docs  = res['documents'][0]
                db_metas = res['metadatas'][0] if 'metadatas' in res else [None] * len(db_ids)
                db_srcs  = ['Les D\u00e9bats'] * len(db_ids)
            except Exception as e:
                # Empty result (e.g. filter too strict)
                db_ids, db_dists, db_docs, db_metas, db_srcs = [], [], [], [], []

        # ---- Combine + optional rerank ----
        all_ids   = np_ids   + db_ids
        all_dists = np_dists + db_dists
        all_docs  = np_docs  + db_docs
        all_srcs  = np_srcs  + db_srcs
        all_metas = np_metas + db_metas

        if rerank_fn is not None and len(all_ids) > 0:
            r_ids, r_dists, _, r_srcs = rerank_fn(all_ids, all_dists, all_metas, all_srcs)
            id_to_doc = dict(zip(all_ids, all_docs))
            r_docs = [id_to_doc.get(rid, '') for rid in r_ids]
        else:
            combined = sorted(
                zip(all_ids, all_dists, all_docs, all_srcs),
                key=lambda x: x[1]
            )
            r_ids   = [x[0] for x in combined]
            r_dists = [x[1] for x in combined]
            r_docs  = [x[2] for x in combined]
            r_srcs  = [x[3] for x in combined]

        rows.append({
            'unique_id':       uid,
            'gold_ids':        row['gold_ids'],
            'predicted_class': row.get('predicted_class', ''),
            'retrieved_ids':   r_ids,
            'distances':       r_dists,
            'documents':       r_docs,
            'sources':         r_srcs,
        })

    return pd.DataFrame(rows)


def _serialize_list(val):
    return val if isinstance(val, list) else json.loads(val)


def save_metadata_df(df, strat_name):
    path = os.path.join(META_CACHE_DIR, f'meta_{strat_name}.jsonl')
    with open(path, 'w', encoding='utf-8') as f:
        for _, row in df.iterrows():
            record = {
                'unique_id':       row['unique_id'],
                'gold_ids':        _serialize_list(row['gold_ids']),
                'predicted_class': row.get('predicted_class', ''),
                'retrieved_ids':   _serialize_list(row['retrieved_ids']),
                'distances':       _serialize_list(row['distances']),
                'documents':       _serialize_list(row['documents']),
                'sources':         _serialize_list(row['sources']),
            }
            f.write(json.dumps(record, ensure_ascii=False) + '\n')


def load_metadata_df(strat_name):
    path = os.path.join(META_CACHE_DIR, f'meta_{strat_name}.jsonl')
    with open(path, encoding='utf-8') as f:
        rows = [json.loads(line) for line in f if line.strip()]
    df = pd.DataFrame(rows)
    for col in ['gold_ids', 'retrieved_ids', 'distances', 'documents', 'sources']:
        if col in df.columns and len(df) > 0 and isinstance(df[col].iloc[0], str):
            df[col] = df[col].apply(json.loads)
    return df


print('Helper functions defined.')

Helper functions defined.


In [9]:
print('Loading universal baseline JSONL...')
base_df = load_universal_baseline_df()
print(f'  {len(base_df)} questions, columns: {base_df.columns.tolist()}')

metadata_dfs = {}

for strat_name, (where_filter, needs_meta, rerank_fn) in FILTER_STRATEGIES.items():
    cache_path = os.path.join(META_CACHE_DIR, f'meta_{strat_name}.jsonl')
    if os.path.exists(cache_path):
        metadata_dfs[strat_name] = load_metadata_df(strat_name)
        print(f'  {strat_name}: loaded from cache ({len(metadata_dfs[strat_name])} rows)')
    elif strat_name == 'baseline':
        # Baseline = universal JSONL, no re-query needed
        metadata_dfs[strat_name] = base_df.copy()
        save_metadata_df(base_df, strat_name)
        print(f'  baseline: copied from universal JSONL and cached')
    else:
        print(f'  {strat_name}: building (re-querying local ChromaDB)...')
        df_built = build_metadata_df(strat_name, where_filter, needs_meta, rerank_fn, base_df)
        save_metadata_df(df_built, strat_name)
        metadata_dfs[strat_name] = df_built
        print(f'    -> saved {len(df_built)} rows')

# Merge source_1 / source_2 (gold source labels) from questions_df into every metadata_df.
# compute_coverage_row_list requires these columns to look up gold documents per source.
uid_to_sources = questions_df[['unique_id', 'source_1', 'source_2']].copy()
for sname in metadata_dfs:
    metadata_dfs[sname] = metadata_dfs[sname].merge(uid_to_sources, on='unique_id', how='left')

print(f'\nAll {len(metadata_dfs)} metadata strategy DataFrames ready.')
print(f'Columns: {metadata_dfs[BASELINE_METADATA].columns.tolist()}')

Loading universal baseline JSONL...
  885 questions, columns: ['question', 'source_1', 'source_2', 'gold_ids', 'id_1', 'question_index', 'id_2', 'unique_id', 'question_type', 'retrieved_ids', 'distances', 'documents', 'sources', 'predicted_class']
  baseline: loaded from cache (885 rows)
  exclude_noisy: loaded from cache (885 rows)
  debates_only: loaded from cache (885 rows)
  debates_and_legal: loaded from cache (885 rows)
  min_speakers_2: loaded from cache (885 rows)
  min_length_500: loaded from cache (885 rows)
  exclude_noisy_min_length: loaded from cache (885 rows)
  rerank_type_boost: loaded from cache (885 rows)
  rerank_speaker_boost: loaded from cache (885 rows)
  rerank_combined: loaded from cache (885 rows)
  exclude_noisy__rerank_speaker: loaded from cache (885 rows)
  debates_legal__rerank_speaker: loaded from cache (885 rows)
  debates_legal__rerank_combined: loaded from cache (885 rows)
  triple_filter__rerank_combined: loaded from cache (885 rows)

All 14 metadata s

## 6. Score Metadata Strategies per Cluster (Training Set)

For each cluster × metadata strategy × routing strategy, compute **Coverage@3**
on the training questions assigned to that cluster.  This yields
`cluster_scores_by_routing[rk][cid][strat_name]`, one aligned score map per routing.

In [10]:
cluster_scores_by_routing = {
    rk: {cid: {} for cid in cluster_train_qids}
    for rk in ROUTING_CONFIGS
}

print(f'Scoring {len(METADATA_STRATEGY_IDS)} strategies '
      f'x {len(cluster_train_qids)} clusters '
      f'x {len(ROUTING_CONFIGS)} routings...')

for strat_name in METADATA_STRATEGY_IDS:
    strat_df   = metadata_dfs[strat_name]
    strat_uids = set(strat_df['unique_id'].tolist())

    for rk, rcfg in ROUTING_CONFIGS.items():
        # Apply routing ONCE on the full strategy df, then slice per cluster
        routed_full = rcfg['fn'](strat_df.copy(), n_results=SCORE_K, **rcfg['kwargs'])

        for cid, qids in cluster_train_qids.items():
            valid_qids = qids & strat_uids
            subset = routed_full[routed_full['unique_id'].isin(valid_qids)]
            if len(subset) == 0:
                cluster_scores_by_routing[rk][cid][strat_name] = float('nan')
                continue
            cov = compute_coverage_row_list(
                subset, k=SCORE_K, collections_dict=gold_collections,
                docs_col=rcfg['docs_col'], sources_col=rcfg['sources_col'],
            )
            cluster_scores_by_routing[rk][cid][strat_name] = float(cov[2].mean())

    print(f'  {strat_name} done')

# Backward-compat alias
cluster_scores = cluster_scores_by_routing['no_rerouting']
print('Done.')

Scoring 14 strategies x 21 clusters x 4 routings...


  baseline done


  exclude_noisy done


  debates_only done


  debates_and_legal done


...

  rerank_combined done


  exclude_noisy__rerank_speaker done


  debates_legal__rerank_speaker done


  debates_legal__rerank_combined done


  triple_filter__rerank_combined done
Done.


## 7. Best Metadata Strategy per Cluster

In [11]:
cluster_to_metadata_by_routing = {}
for rk in ROUTING_CONFIGS:
    cmap = {}
    for cid, scores in cluster_scores_by_routing[rk].items():
        valid = {s: v for s, v in scores.items() if not np.isnan(v)}
        cmap[cid] = max(valid, key=valid.get) if valid else BASELINE_METADATA
    cluster_to_metadata_by_routing[rk] = cmap

# Backward-compat alias
cluster_to_metadata = cluster_to_metadata_by_routing['no_rerouting']

print('Clusters where routing changes the optimal metadata strategy vs no_rerouting:\n')
for rk in ['qre', 'ums', 'qre_ums']:
    n_changed = sum(
        cluster_to_metadata_by_routing['no_rerouting'][cid] !=
        cluster_to_metadata_by_routing[rk][cid]
        for cid in cluster_train_qids
    )
    print(f'  no_rerouting -> {rk}: {n_changed}/{len(cluster_train_qids)} clusters change')

print()
summary_rows = []
for cid in sorted(cluster_to_metadata):
    n_q = len(cluster_train_qids[cid])
    best = cluster_to_metadata[cid]
    best_score     = cluster_scores[cid].get(best, float('nan'))
    baseline_score = cluster_scores[cid].get(BASELINE_METADATA, float('nan'))
    summary_rows.append({
        'cluster':             cid,
        'n_train':             n_q,
        'best_metadata':       best,
        'coverage@3_best':     round(best_score, 3),
        'coverage@3_baseline': round(baseline_score, 3),
        'delta':               round(best_score - baseline_score, 3),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df.sort_values('n_train', ascending=False))

Clusters where routing changes the optimal metadata strategy vs no_rerouting:

  no_rerouting -> qre: 3/21 clusters change
  no_rerouting -> ums: 5/21 clusters change
  no_rerouting -> qre_ums: 7/21 clusters change



,cluster,n_train,best_metadata,coverage@3_best,coverage@3_baseline,delta
1,0,149,exclude_noisy,0.493,0.158,0.336
0,-1,123,baseline,0.492,0.492,0.000
4,3,79,exclude_noisy,0.424,0.101,0.323
6,5,74,baseline,0.527,0.527,0.000
15,14,37,baseline,0.378,0.378,0.000
13,12,27,baseline,0.389,0.389,0.000
9,8,21,baseline,0.524,0.524,0.000
20,19,17,baseline,0.353,0.353,0.000
5,4,15,baseline,0.567,0.567,0.000
18,17,14,baseline,0.500,0.500,0.000


In [12]:
# Re-score with per-routing score_k (parity with C2 v2 fix).
# UMS / QRe+UMS force equal distribution across <=3 collections, so SCORE_K=3
# leaves only 1 slot per collection — a binary hit/miss signal that is too noisy
# for picking the best metadata strategy. score_k=6 -> 2 slots/collection,
# restoring per-collection resolution comparable to no_rerouting@3.

SCORE_K_PER_ROUTING = {'no_rerouting': 3, 'qre': 3, 'ums': 6, 'qre_ums': 6}

cluster_scores_by_routing_v2 = {
    rk: {cid: {} for cid in cluster_train_qids}
    for rk in ROUTING_CONFIGS
}

print('Re-scoring with per-routing score_k:', SCORE_K_PER_ROUTING)
for strat_name in METADATA_STRATEGY_IDS:
    strat_df   = metadata_dfs[strat_name]
    strat_uids = set(strat_df['unique_id'].tolist())

    for rk, rcfg in ROUTING_CONFIGS.items():
        rk_score_k = SCORE_K_PER_ROUTING[rk]
        routed_full = rcfg['fn'](strat_df.copy(), n_results=rk_score_k, **rcfg['kwargs'])

        for cid, qids in cluster_train_qids.items():
            valid_qids = qids & strat_uids
            subset = routed_full[routed_full['unique_id'].isin(valid_qids)]
            if len(subset) == 0:
                cluster_scores_by_routing_v2[rk][cid][strat_name] = float('nan')
                continue
            cov = compute_coverage_row_list(
                subset, k=rk_score_k, collections_dict=gold_collections,
                docs_col=rcfg['docs_col'], sources_col=rcfg['sources_col'],
            )
            cluster_scores_by_routing_v2[rk][cid][strat_name] = float(cov[2].mean())

    print(f'  {strat_name} done')

# Overwrite so all downstream cells pick up the corrected scores
cluster_scores_by_routing = cluster_scores_by_routing_v2
cluster_scores = cluster_scores_by_routing['no_rerouting']

# Re-compute best metadata strategy per cluster
cluster_to_metadata_by_routing = {}
for rk in ROUTING_CONFIGS:
    cmap = {}
    for cid, scores in cluster_scores_by_routing[rk].items():
        valid = {s: v for s, v in scores.items() if not np.isnan(v)}
        cmap[cid] = max(valid, key=valid.get) if valid else BASELINE_METADATA
    cluster_to_metadata_by_routing[rk] = cmap
cluster_to_metadata = cluster_to_metadata_by_routing['no_rerouting']

print('\nClusters where routing changes the optimal metadata strategy (v2):')
for rk in ['qre', 'ums', 'qre_ums']:
    n_ch = sum(
        cluster_to_metadata_by_routing['no_rerouting'][cid] !=
        cluster_to_metadata_by_routing[rk][cid]
        for cid in cluster_train_qids
    )
    print(f'  no_rerouting -> {rk}: {n_ch}/{len(cluster_train_qids)}')

print('\nFraction of clusters where best == baseline (v2):')
for rk in ROUTING_CONFIGS:
    n_same = sum(cluster_to_metadata_by_routing[rk][cid] == BASELINE_METADATA for cid in cluster_train_qids)
    print(f'  {rk:<20s}: {n_same}/{len(cluster_train_qids)} ({100*n_same/len(cluster_train_qids):.0f}%)')
print('Done. cluster_scores_by_routing updated.')


Re-scoring with per-routing score_k: {'no_rerouting': 3, 'qre': 3, 'ums': 6, 'qre_ums': 6}


  baseline done


  exclude_noisy done


  debates_only done


  debates_and_legal done


...

  rerank_combined done


  exclude_noisy__rerank_speaker done


  debates_legal__rerank_speaker done


  debates_legal__rerank_combined done


  triple_filter__rerank_combined done

Clusters where routing changes the optimal metadata strategy (v2):
  no_rerouting -> qre: 3/21
  no_rerouting -> ums: 4/21
  no_rerouting -> qre_ums: 6/21

Fraction of clusters where best == baseline (v2):
  no_rerouting        : 15/21 (71%)
  qre                 : 18/21 (86%)
  ums                 : 17/21 (81%)
  qre_ums             : 19/21 (90%)
Done. cluster_scores_by_routing updated.


## 8. Assign Test Questions to Nearest Cluster

Project test embeddings into the training UMAP space (10D) and find the nearest
centroid.  Each routing gets its own `assigned_metadata_{rk}` column derived from
its own `cluster_to_metadata_by_routing[rk]` map.

In [13]:
X_test = np.array(test_df['embeddings'].tolist())
X_test_umap = reducer.transform(X_test)  # use 10D reducer from clustering

cluster_ids_ordered = sorted(centroids.keys())
centroid_matrix     = np.array([centroids[cid] for cid in cluster_ids_ordered])

dists        = euclidean_distances(X_test_umap, centroid_matrix)
nearest_idx  = dists.argmin(axis=1)
assigned_clusters = [cluster_ids_ordered[i] for i in nearest_idx]

test_df = test_df.copy()
test_df['assigned_cluster'] = assigned_clusters

for rk in ROUTING_CONFIGS:
    col = f'assigned_metadata_{rk}'
    test_df[col] = [
        cluster_to_metadata_by_routing[rk].get(cid, BASELINE_METADATA)
        for cid in assigned_clusters
    ]

# Backward-compat alias
test_df['assigned_metadata'] = test_df['assigned_metadata_no_rerouting']

print('Metadata strategy distribution by routing (test set):')
for rk, rcfg in ROUTING_CONFIGS.items():
    print(f"\n  [{rcfg['label']}]")
    print(test_df[f'assigned_metadata_{rk}'].value_counts().to_string())

Metadata strategy distribution by routing (test set):

  [No Rerouting]
assigned_metadata_no_rerouting
baseline             129
exclude_noisy         82
rerank_type_boost      8

  [Query Rerouting (QRe)]
assigned_metadata_qre
baseline             203
exclude_noisy          8
rerank_type_boost      8

  [UMS (Forced Equal)]
assigned_metadata_ums
baseline                121
exclude_noisy            78
rerank_speaker_boost     20

  [QRe + UMS]
assigned_metadata_qre_ums
baseline                191
rerank_speaker_boost     20
exclude_noisy             8


## 9. Build Evaluation Conditions (8 Conditions)

**8 conditions** = 4 routing strategies × 2 metadata policies.

| | **No Rerouting** | **QRe** | **UMS** | **QRe + UMS** |
|---|---|---|---|---|
| **Global Baseline** (uniform metadata) | ✓ | ✓ | ✓ | ✓ |
| **Task-Conditioned** (cluster-aligned) | ✓ | ✓ | ✓ | ✓ |

**Key principle:** each TC condition uses the metadata map *trained under the same routing*
applied at test time.

In [14]:
def load_condition_raw_meta(df, metadata_col):
    """Concatenate subsets from each metadata_dfs entry for questions in that strategy bucket."""
    chunks = []
    for strat_name in df[metadata_col].unique():
        strat_df = metadata_dfs[strat_name]
        qids     = df[df[metadata_col] == strat_name]['unique_id'].tolist()
        subset   = strat_df[strat_df['unique_id'].isin(qids)]
        chunks.append(subset)
    return pd.concat(chunks, ignore_index=True)


test_df['baseline_metadata'] = BASELINE_METADATA

print('Loading raw data for all conditions...')
raw_baseline = load_condition_raw_meta(test_df, 'baseline_metadata')

raw_tc = {}
for rk in ROUTING_CONFIGS:
    raw_tc[rk] = load_condition_raw_meta(test_df, f'assigned_metadata_{rk}')
    print(f'  TC/{rk}: {len(raw_tc[rk])} rows loaded')

# CONDITION_CONFIGS: key -> (raw_df, routing_fn, routing_kwargs, docs_col, sources_col, label)
CONDITION_CONFIGS = {}
for rk, rcfg in ROUTING_CONFIGS.items():
    CONDITION_CONFIGS[f'baseline_{rk}'] = (
        raw_baseline.copy(), rcfg['fn'], rcfg['kwargs'],
        rcfg['docs_col'], rcfg['sources_col'],
        f"Global Baseline + {rcfg['label']}",
    )
    CONDITION_CONFIGS[f'tc_{rk}'] = (
        raw_tc[rk].copy(), rcfg['fn'], rcfg['kwargs'],
        rcfg['docs_col'], rcfg['sources_col'],
        f"Task-Conditioned + {rcfg['label']}",
    )

print(f'\n{len(CONDITION_CONFIGS)} conditions:')
for key, (*_, lbl) in CONDITION_CONFIGS.items():
    print(f'  [{key}] {lbl}')

Loading raw data for all conditions...
  TC/no_rerouting: 219 rows loaded
  TC/qre: 219 rows loaded
  TC/ums: 219 rows loaded
  TC/qre_ums: 219 rows loaded

8 conditions:
  [baseline_no_rerouting] Global Baseline + No Rerouting
  [tc_no_rerouting] Task-Conditioned + No Rerouting
  [baseline_qre] Global Baseline + Query Rerouting (QRe)
  [tc_qre] Task-Conditioned + Query Rerouting (QRe)
  [baseline_ums] Global Baseline + UMS (Forced Equal)
  [tc_ums] Task-Conditioned + UMS (Forced Equal)
  [baseline_qre_ums] Global Baseline + QRe + UMS
  [tc_qre_ums] Task-Conditioned + QRe + UMS


## 10. Results

In [15]:
results_rows = []

for cond_key, (raw_df, routing_fn, routing_kwargs, docs_col, sources_col, display_label) in CONDITION_CONFIGS.items():
    row = {'condition': cond_key, 'label': display_label}
    for k in K_VALUES:
        df_k = routing_fn(raw_df.copy(), n_results=k, **routing_kwargs)

        cov = compute_coverage_row_list(
            df_k, k=k, collections_dict=gold_collections,
            docs_col=docs_col, sources_col=sources_col,
        )
        row[f'coverage@{k}'] = round(float(cov[2].mean()), 3)

        recall = compute_recall_metrics_dataframe(df_k, k_values=[k])
        row[f'recall@{k}'] = round(float(np.mean(recall[f'recall@{k}'])), 3)

    results_rows.append(row)
    print(f'  {display_label} done')

results_df    = pd.DataFrame(results_rows)
results_table = results_df.set_index('condition')

coverage_cols = [f'coverage@{k}' for k in K_VALUES]
recall_cols   = [f'recall@{k}'   for k in K_VALUES]

display_table = results_df.set_index('label')[coverage_cols + recall_cols]
display(display_table.style.highlight_max(axis=0, color='#c6efce').format('{:.3f}'))

print('\n\u0394 Coverage@k and Recall@k  (Task-Conditioned \u2212 Global Baseline):')
delta_rows = []
for rk, rcfg in ROUTING_CONFIGS.items():
    b = results_table.loc[f'baseline_{rk}', coverage_cols + recall_cols]
    t = results_table.loc[f'tc_{rk}',       coverage_cols + recall_cols]
    delta_rows.append({'Routing': rcfg['label'], **(t - b).round(3).to_dict()})
delta_df = pd.DataFrame(delta_rows).set_index('Routing')
display(delta_df.style.background_gradient(cmap='RdYlGn', axis=None).format('{:+.3f}'))

100%|██████████| 219/219 [00:00<00:00, 19899.32it/s]


  Global Baseline + No Rerouting done


100%|██████████| 219/219 [00:00<00:00, 24286.20it/s]


  Task-Conditioned + No Rerouting done


100%|██████████| 219/219 [00:00<00:00, 18245.16it/s]


...

100%|██████████| 219/219 [00:00<00:00, 16417.09it/s]


  Task-Conditioned + QRe + UMS done


,coverage@3,coverage@5,coverage@10,recall@3,recall@5,recall@10
label,,,,,,
Global Baseline + No Rerouting,0.390,0.457,0.571,0.390,0.457,0.573
Task-Conditioned + No Rerouting,0.498,0.573,0.680,0.502,0.575,0.683
Global Baseline + Query Rerouting (QRe),0.495,0.578,0.687,0.500,0.580,0.689
Task-Conditioned + Query Rerouting (QRe),0.505,0.582,0.689,0.509,0.584,0.692
Global Baseline + UMS (Forced Equal),0.534,0.598,0.715,0.534,0.600,0.717
Task-Conditioned + UMS (Forced Equal),0.559,0.607,0.731,0.562,0.610,0.733
Global Baseline + QRe + UMS,0.546,0.607,0.731,0.548,0.610,0.733
Task-Conditioned + QRe + UMS,0.546,0.607,0.726,0.548,0.610,0.728



Δ Coverage@k and Recall@k  (Task-Conditioned − Global Baseline):


,coverage@3,coverage@5,coverage@10,recall@3,recall@5,recall@10
Routing,,,,,,
No Rerouting,+0.108,+0.116,+0.109,+0.112,+0.118,+0.110
Query Rerouting (QRe),+0.010,+0.004,+0.002,+0.009,+0.004,+0.003
UMS (Forced Equal),+0.025,+0.009,+0.016,+0.028,+0.010,+0.016
QRe + UMS,+0.000,+0.000,-0.005,+0.000,+0.000,-0.005


In [16]:
# ── Compact summary: Coverage@k — Baseline vs TC per routing strategy ────────
print("Coverage@k — Global Baseline vs Task-Conditioned\n")
header = f"{'Routing':<22} {'Chunking':<20} {'C@3':>6} {'C@5':>6} {'C@10':>7}"
print(header)
print("-" * len(header))
for rk, rcfg in ROUTING_CONFIGS.items():
    b_row = results_table.loc[f"baseline_{rk}"]
    t_row = results_table.loc[f"tc_{rk}"]
    for cond, row in [("Global Baseline", b_row), ("Task-Conditioned", t_row)]:
        print(f"  {rcfg['label']:<20} {cond:<20} "
              f"{row['coverage@3']:>6.3f} {row['coverage@5']:>6.3f} {row['coverage@10']:>7.3f}")
    delta = t_row[coverage_cols] - b_row[coverage_cols]
    sign = lambda v: f"+{v:.3f}" if v >= 0 else f"{v:.3f}"
    print(f"  {'':20} {'Δ':<20} "
          f"{sign(delta['coverage@3']):>6} {sign(delta['coverage@5']):>6} {sign(delta['coverage@10']):>7}")
    print()


Coverage@k — Global Baseline vs Task-Conditioned

Routing                Chunking                C@3    C@5    C@10
-----------------------------------------------------------------
  No Rerouting         Global Baseline       0.390  0.457   0.571
  No Rerouting         Task-Conditioned      0.498  0.573   0.680
                       Δ                    +0.108 +0.116  +0.109

  Query Rerouting (QRe) Global Baseline       0.495  0.578   0.687
  Query Rerouting (QRe) Task-Conditioned      0.505  0.582   0.689
                       Δ                    +0.010 +0.004  +0.002

  UMS (Forced Equal)   Global Baseline       0.534  0.598   0.715
  UMS (Forced Equal)   Task-Conditioned      0.559  0.607   0.731
                       Δ                    +0.025 +0.009  +0.016

  QRe + UMS            Global Baseline       0.546  0.607   0.731
  QRe + UMS            Task-Conditioned      0.546  0.607   0.726
                       Δ                    +0.000 +0.000  -0.005



In [18]:
test_df

,question,embeddings,gold_ids,question_type,id_2,question_index,source_1,unique_id,id_1,source_2,assigned_cluster,assigned_metadata_no_rerouting,assigned_metadata_qre,assigned_metadata_ums,assigned_metadata_qre_ums,assigned_metadata,baseline_metadata
0,Comment M. Blanc justifie-t-il l'importance de...,"[-0.0022237906232476234, 0.04156947508454323, ...","[18870112, 1887-01-11_004_000.txt]",MH - Generic,1887-01-11_004_000.txt,1,Le Gaulois,1_18870112_1887-01-11_004_000.txt_1,18870112,Les Débats,8,baseline,baseline,baseline,baseline,baseline,baseline
1,Quelle position M. Goblet adopte-t-il face à l...,"[0.006911382079124451, 0.022054297849535942, 0...","[18870118, 1887-01-17_016_002.txt]",MH - Generic,1887-01-17_016_002.txt,1,Le Gaulois,4_18870118_1887-01-17_016_002.txt_1,18870118,Les Débats,16,baseline,baseline,baseline,baseline,baseline,baseline
2,Comment M. Achard justifie-t-il l'urgence de r...,"[0.04253409802913666, 0.07026002556085587, 0.0...","[18870118, 1887-01-17_016_002.txt]",MH - Generic,1887-01-17_016_002.txt,2,Le Gaulois,5_18870118_1887-01-17_016_002.txt_2,18870118,Les Débats,16,baseline,baseline,baseline,baseline,baseline,baseline
3,"Selon le baron de Mackau, quelles sont les con...","[-0.06214575842022896, 0.00920034945011139, 0....","[18870125, 1887-01-24_002_003.txt]",MH - Generic,1887-01-24_002_003.txt,1,Le Gaulois,10_18870125_1887-01-24_002_003.txt_1,18870125,Les Débats,5,baseline,baseline,baseline,baseline,baseline,baseline
4,Quelle est la perception de la presse concerna...,"[-0.0033271650318056345, 0.0105939581990242, -...","[18870126, 1887-01-25_006_003.txt]",MH - Generic,1887-01-25_006_003.txt,1,Le Gaulois,13_18870126_1887-01-25_006_003.txt_1,18870126,Les Débats,12,baseline,baseline,baseline,baseline,baseline,baseline
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
214,En quoi les deux journaux diffèrent-ils dans l...,"[-0.05277420952916145, 0.004449712112545967, -...","[18870125, 18870126]",MH - Generic,18870126,1,Le Gaulois,852_18870125_18870126_1,18870125,L'Intransigeant,3,exclude_noisy,baseline,exclude_noisy,baseline,exclude_noisy,baseline
215,"En 1887, comment L'Intransigeant et Le Gaulois...","[-0.019408350810408592, 0.039293959736824036, ...","[18870311, 18870314]",MH - Generic,18870314,2,Le Gaulois,856_18870311_18870314_2,18870311,L'Intransigeant,0,exclude_noisy,baseline,exclude_noisy,baseline,exclude_noisy,baseline
216,"En 1887, comment L’Intransigeant et Le Gaulois...","[-0.013041038066148758, 0.0025937978643924, -0...","[18870130, 18870131]",MH - Generic,18870131,0,Le Gaulois,862_18870130_18870131_0,18870130,L'Intransigeant,0,exclude_noisy,baseline,exclude_noisy,baseline,exclude_noisy,baseline
217,"En 1887, comment L’Intransigeant et Le Gaulois...","[-0.008412779308855534, 0.028527285903692245, ...","[18870511, 18870512]",MH - Generic,18870512,0,Le Gaulois,870_18870511_18870512_0,18870511,L'Intransigeant,0,exclude_noisy,baseline,exclude_noisy,baseline,exclude_noisy,baseline


In [19]:
RK = "no_rerouting"
K = 3
rcfg = ROUTING_CONFIGS[RK]

# Per-question Coverage@K — baseline (uniform 10000_hierarchical)
b_routed = rcfg["fn"](raw_baseline.copy(), n_results=K, **rcfg["kwargs"])
b_cov = compute_coverage_row_list(
    b_routed, k=K, collections_dict=gold_collections,
    docs_col=rcfg["docs_col"], sources_col=rcfg["sources_col"],
)
b_per_q = pd.Series(np.asarray(b_cov[2]), index=b_routed["unique_id"].values, name="cov_baseline")

# Per-question Coverage@K — task-conditioned (per-cluster optimal strategy)
t_routed = rcfg["fn"](raw_tc[RK].copy(), n_results=K, **rcfg["kwargs"])
t_cov = compute_coverage_row_list(
    t_routed, k=K, collections_dict=gold_collections,
    docs_col=rcfg["docs_col"], sources_col=rcfg["sources_col"],
)
t_per_q = pd.Series(np.asarray(t_cov[2]), index=t_routed["unique_id"].values, name="cov_tc")

# Join with test_df metadata (cluster + assigned strategy under this routing)
gain_df = (
    test_df[["unique_id", "assigned_cluster", f"assigned_metadata_{RK}"]]
    .rename(columns={f"assigned_strategy_{RK}": "assigned_strategy"})
    .set_index("unique_id")
    .join(b_per_q)
    .join(t_per_q)
    .dropna(subset=["cov_baseline", "cov_tc"])
)
gain_df["delta"] = gain_df["cov_tc"] - gain_df["cov_baseline"]

print(f"Test questions scored: {len(gain_df)}")
print(f"Overall Coverage@{K}  baseline={gain_df['cov_baseline'].mean():.3f}  "
      f"tc={gain_df['cov_tc'].mean():.3f}  Δ={gain_df['delta'].mean():+.3f}")

  0%|          | 0/219 [00:00<?, ?it/s]

Test questions scored: 219
Overall Coverage@3  baseline=0.390  tc=0.498  Δ=+0.107


In [22]:
gain_df

,assigned_cluster,assigned_metadata_no_rerouting,cov_baseline,cov_tc,delta
unique_id,,,,,
1_18870112_1887-01-11_004_000.txt_1,8,baseline,0.0,0.0,0.0
4_18870118_1887-01-17_016_002.txt_1,16,baseline,0.5,0.5,0.0
5_18870118_1887-01-17_016_002.txt_2,16,baseline,0.5,0.5,0.0
10_18870125_1887-01-24_002_003.txt_1,5,baseline,1.0,1.0,0.0
13_18870126_1887-01-25_006_003.txt_1,12,baseline,1.0,1.0,0.0
...,...,...,...,...,...
852_18870125_18870126_1,3,exclude_noisy,0.0,0.5,0.5
856_18870311_18870314_2,0,exclude_noisy,0.5,0.5,0.0
862_18870130_18870131_0,0,exclude_noisy,0.0,0.5,0.5


In [24]:
# ── Gain by cluster ──────────────────────────────────────────────────────────
by_cluster = (
    gain_df.groupby("assigned_cluster")
           .agg(n_test=("delta", "size"),
                strategy=("assigned_metadata_no_rerouting", "first"),
                cov_baseline=("cov_baseline", "mean"),
                cov_tc=("cov_tc", "mean"),
                delta=("delta", "mean"))
           .round(3)
           .sort_values("delta", ascending=False)
)
print("Gain by cluster (Naive Retriever, Coverage@3):\n")
print(by_cluster.to_string())

n_pos = int((by_cluster["delta"] > 0).sum())
n_neg = int((by_cluster["delta"] < 0).sum())
n_zero = int((by_cluster["delta"] == 0).sum())
print(f"\n  Clusters with gain:      {n_pos}")
print(f"  Clusters with loss:      {n_neg}")
print(f"  Clusters unchanged:      {n_zero}")

Gain by cluster (Naive Retriever, Coverage@3):

                  n_test           strategy  cov_baseline  cov_tc  delta
assigned_cluster                                                        
 0                    43      exclude_noisy         0.151   0.500  0.349
 3                    27      exclude_noisy         0.167   0.444  0.278
 18                    8      exclude_noisy         0.188   0.438  0.250
-1                     5           baseline         0.700   0.700  0.000
 17                   10           baseline         0.400   0.400  0.000
 16                   14           baseline         0.464   0.464  0.000
 15                    1           baseline         0.500   0.500  0.000
 14                   20           baseline         0.475   0.475  0.000
 13                    2           baseline         0.500   0.500  0.000
 12                   16           baseline         0.406   0.406  0.000
 10                    3           baseline         0.833   0.833  0.000
 9 

In [26]:
# ── Gain by winning strategy ─────────────────────────────────────────────────
by_strategy = (
    gain_df.groupby("assigned_metadata_no_rerouting")
           .agg(n_test=("delta", "size"),
                n_clusters=("assigned_cluster", "nunique"),
                cov_baseline=("cov_baseline", "mean"),
                cov_tc=("cov_tc", "mean"),
                delta=("delta", "mean"))
           .round(3)
           .sort_values("delta", ascending=False)
)
print("Gain by winning strategy (Naive Retriever, Coverage@3):\n")
print(by_strategy.to_string())

Gain by winning strategy (Naive Retriever, Coverage@3):

                                n_test  n_clusters  cov_baseline  cov_tc  delta
assigned_metadata_no_rerouting                                                 
exclude_noisy                       82           4         0.183   0.470  0.287
baseline                           129          15         0.516   0.516  0.000
rerank_type_boost                    8           1         0.500   0.500  0.000
